# Purpose

## Load the merged county-year dataset and engineer analysis-ready features (YoY growth, boom intensity, slowdown, volatility) for modeling and clustering.

# Imports

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Set data paths
PROJECT_ROOT = Path().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
# Import datasets
merged_path = DATA_PROCESSED / "merged_fl_county_year.csv"
df = pd.read_csv(merged_path)

# Review Dataset

In [4]:
df.shape

(335, 9)

In [5]:
df.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   fips           335 non-null    int64  
 1   RegionName     335 non-null    object 
 2   year           335 non-null    int64  
 3   zhvi_annual    335 non-null    float64
 4   STNAME         335 non-null    object 
 5   CTYNAME        335 non-null    object 
 6   domestic_mig   335 non-null    int64  
 7   population     335 non-null    int64  
 8   rdomestic_mig  268 non-null    float64
dtypes: float64(2), int64(4), object(3)
memory usage: 23.7+ KB


In [7]:
# Change FIPS to string
df["fips"] = df["fips"].astype(str)

In [8]:
df["fips"].nunique() # Expect 67 counties

67

In [9]:
df["year"].unique()

array([2020, 2021, 2022, 2023, 2024])

In [10]:
# Ensure sorted
df = df.sort_values(["fips", "year"]).reset_index(drop=True)

In [11]:
df.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467


# Feature: Year Over Year Growth

In [13]:
# Calculate percentage change in annual zhvi per county (FIPS)
df["yoy_growth"] = df.groupby("fips")["zhvi_annual"].pct_change()

In [14]:
df.head()

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig,yoy_growth
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN,NaN
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857,0.142634
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050,0.154966
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100,0.047640
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442,0.029851


# Feature: Boom intensity

## “How much did prices rise during the surge period (2020 - 2022)?”

### Computed as: (zhvi_2022 / zhvi_2020) - 1

In [15]:
# Make 2020 ZHVI dataframe per FIPS (county)
zhvi_2020 = df[df["year"] == 2020][["fips", "zhvi_annual"]].rename(columns={"zhvi_annual": "zhvi_2020"})

In [16]:
# Make 2022 ZHVI dataframe per FIPS (county)
zhvi_2022 = df[df["year"] == 2022][["fips", "zhvi_annual"]].rename(columns={"zhvi_annual": "zhvi_2022"})

In [17]:
# Merge dataframes
boom = zhvi_2020.merge(zhvi_2022, on="fips", how="inner")

In [18]:
# Create boom growth percentage column
boom["boom_growth_2020_2022"] = (boom["zhvi_2022"] / boom["zhvi_2020"]) - 1

In [19]:
boom.head()

,fips,zhvi_2020,zhvi_2022,boom_growth_2020_2022
0,12001,213860.291033,282232.292655,0.319704
1,12003,214588.831681,293861.137166,0.369415
2,12005,241755.736820,344934.398180,0.426789
3,12007,153594.952120,219482.984261,0.428973
4,12009,235133.577573,342247.395935,0.455545


In [20]:
# Merge subset (fips and boom) to master dataframe
df = df.merge(boom[["fips", "boom_growth_2020_2022"]], on="fips", how="left")

In [21]:
df.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig,yoy_growth,boom_growth_2020_2022
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN,NaN,0.319704
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857,0.142634,0.319704
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050,0.154966,0.319704
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100,0.047640,0.319704
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442,0.029851,0.319704
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN,NaN,0.369415
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018,0.165379,0.369415
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853,0.175081,0.369415
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875,-0.004726,0.369415
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467,0.021193,0.369415


# Feature: Slowdown intensity

## “Did growth slow down after 2022?”

### Computed as: 2024 YoY minus 2022 YoY

In [22]:
# Create 2022 YoY dataframe per FIPS
yoy_2022 = df[df["year"] == 2022][["fips", "yoy_growth"]].rename(columns={"yoy_growth": "yoy_2022"})

In [23]:
# Create 2024 YoY dataframe per FIPS
yoy_2024 = df[df["year"] == 2024][["fips", "yoy_growth"]].rename(columns={"yoy_growth": "yoy_2024"})

In [24]:
# Merge dataframes
slow = yoy_2022.merge(yoy_2024, on="fips", how="inner")

In [25]:
# Compute slowdown
slow["cooling_delta_2024_vs_2022"] = slow["yoy_2024"] - slow["yoy_2022"]

In [26]:
slow.head()

,fips,yoy_2022,yoy_2024,cooling_delta_2024_vs_2022
0,12001,0.154966,0.029851,-0.125115
1,12003,0.175081,0.021193,-0.153888
2,12005,0.228471,-0.010958,-0.239428
3,12007,0.215059,0.033148,-0.181912
4,12009,0.233674,0.005254,-0.228420


In [27]:
# Merge with master dataframe
df = df.merge(slow[["fips", "cooling_delta_2024_vs_2022"]], on="fips", how="left")

# Feature: Volatility

## "How stable is this county's growth?"

### Calculated using standard deviation of YoY growth by county (2021–2024).

In [28]:
# Create volatility dataframe per FIPS
vol = (df[df["year"] >= 2021].groupby("fips", as_index=False)["yoy_growth"].std().rename(columns={"yoy_growth": "yoy_volatility"}))

In [29]:
vol.head()

,fips,yoy_volatility
0,12001,0.064152
1,12003,0.094209
2,12005,0.108733
3,12007,0.090090
4,12009,0.111754


In [30]:
# Merge with master dataframe
df = df.merge(vol, on="fips", how="left")

In [31]:
df.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig,yoy_growth,boom_growth_2020_2022,cooling_delta_2024_vs_2022,yoy_volatility
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN,NaN,0.319704,-0.125115,0.064152
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857,0.142634,0.319704,-0.125115,0.064152
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050,0.154966,0.319704,-0.125115,0.064152
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100,0.047640,0.319704,-0.125115,0.064152
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442,0.029851,0.319704,-0.125115,0.064152
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN,NaN,0.369415,-0.153888,0.094209
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018,0.165379,0.369415,-0.153888,0.094209
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853,0.175081,0.369415,-0.153888,0.094209
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875,-0.004726,0.369415,-0.153888,0.094209
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467,0.021193,0.369415,-0.153888,0.094209


# Exports

In [32]:
df.to_csv("../data/processed/merged_fl_county_year_features.csv", index=False)